In [1]:
import pandas as pd
data = [[1, 1], [2, 1], [3, 1], [4, 2], [5, 1], [6, 2], [7, 2]]
logs = pd.DataFrame(data, columns=['id', 'num']).astype({'id':'Int64', 'num':'Int64'})

In [4]:
from pyspark.sql import SparkSession
spark= SparkSession.builder.appName("consecutiveNumber").getOrCreate()
df1=spark.createDataFrame(logs)
df1.show()

+---+---+
| id|num|
+---+---+
|  1|  1|
|  2|  1|
|  3|  1|
|  4|  2|
|  5|  1|
|  6|  2|
|  7|  2|
+---+---+



In [29]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag,col
winspec=Window.orderBy(col("id"))
df2=df1.withColumn("first_lag",lag("num",1).over(winspec)).withColumn("second_lag",lag("num",2).over(winspec))
df3=df2.select("num").where((col("num")==col("first_lag")) & (col("first_lag")==col("second_lag"))).distinct()
df3.show()


26/06/15 06:26:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/15 06:26:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/15 06:26:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/15 06:26:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+
|num|
+---+
|  1|
+---+



## creating temp view for this data frame

In [32]:
df1.createOrReplaceTempView("consecutive_number")
sqlDF=spark.sql("""
            with abc_cte as (
                select
                id,
                num,
                lag(num,1) over(order by id) as first_lag,
                lag(num,2) over(order by id) as second_lag
                from consecutive_number
                )
                select
                     distinct num
                     from abc_cte
                        where num=first_lag and first_lag=second_lag    
""")
sqlDF.show()

26/06/16 05:46:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 05:46:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+
|num|
+---+
|  1|
+---+



26/06/16 05:47:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/16 05:47:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/06/16 22:31:29 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 126260 ms exceeds timeout 120000 ms
26/06/16 22:31:29 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/16 22:31:35 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$